# (Week3) Day1: Introduction to HuggingFace Models: [Google Colab Notebook Copy]

### Checking the info on the GPU being used:

In [ ]:
# gpu_info = !nvidia-smi
# gpu_info = '\n'.join(gpu_info)
# if gpu_info.find('failed') >= 0:
#   print("Not connected to GPU!!")
# else:
#   print(gpu_info)
#   if gpu_info.find('Tesla T4'):
#     print("Successfully - Connected to a T4")
#   else:
#     print("NOT CONNECTED TO A T$")

## Connecting Hugging Face to Google Colab:

* Create an Access Token for you Hugging Face account throught the settings page in the HuggingFace website.

* Then add the key as a secret variable in the secrets option of the google colab notebook.

* After that, copy the code displayed right underneath the entered secret key for utilising the access token/key  while coding.

In [ ]:
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('HuggingFace')
login(hf_token, add_to_git_credential= True)

### Running a diffuser model:

In [ ]:
from IPython.display import display
from diffusers import AutoPipelineForText2Image
import torch

pipe = AutoPipelineForText2Image.from_pretrained("stabilityai/sdxl-turbo", torch_dtype = torch.float16, variant = "fp16")
pipe.to("cuda")
prompt = "A knight with black armour with golden borders, standing on a cliff looking towards an ominus looking dark castle in the distance. The sky is dark grey with red lightning"
image = pipe(prompt = prompt, num_inference_steps = 4, guidance_scale = 0.0).images[0]
display(image)

### Using a different model but with a 'Refiner' Model

In [ ]:
from diffusers import DiffusionPipeline
import torch

base = DiffusionPipeline.from_pretrained("stabilityai/stable-diffusion-xl-base-1.0", torch_dtype=torch.float16, variant="fp16", use_safetensors=True)
base.to("cuda")
refiner = DiffusionPipeline.from_pretrained("stabilityai/stable-diffusion-xl-refiner-1.0", text_encoder_2=base.text_encoder_2, vae=base.vae, torch_dtype=torch.float16, use_safetensors=True, variant="fp16",)
refiner.to("cuda")

# Define how many steps and what % of steps to be run on each experts (80/20) here
n_steps = 40
high_noise_frac = 0.8

prompt = "A class of data scientists learning AI engineering in a vibrant high-energy pop-art style"
# run both experts
image = base(
    prompt=prompt,
    num_inference_steps=n_steps,
    denoising_end=high_noise_frac,
    output_type="latent",
).images

image = refiner(
    prompt=prompt,
    num_inference_steps=n_steps,
    denoising_start=high_noise_frac,
    image=image,
).images[0]

display(image)

### Trying out Text-To-Speech models:

In [ ]:
!pip install --upgrade datasets==3.6.0

In [ ]:
from transformers import pipeline
from datasets import load_dataset
import soundfile as sf
import torch
from IPython.display import Audio

synthesiser = pipeline("text-to-speech", "microsoft/speecht5_tts", device='cuda')
embeddings_dataset = load_dataset("matthijs/cmu-arctic-xvectors", split="validation", trust_remote_code=True)
speaker_embedding = torch.tensor(embeddings_dataset[7306]["xvector"]).unsqueeze(0)
speech = synthesiser("Hi to an artificial intelligence engineer, on the way to mastery!", forward_params={"speaker_embeddings": speaker_embedding})

Audio(speech["audio"], rate=speech["sampling_rate"])